# import


In [ ]:
from __future__ import annotations

import gc
import html
import os
import re
import warnings
from collections import defaultdict
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import pipeline


## 1 : ตั้งค่าโมเดล ไฟล์นำเข้า และตัวเลือกสำหรับการทดสอบ

In [ ]:
# ส่วนตั้งค่า
# กำหนดตำแหน่งโฟลเดอร์ภาพ, template สำหรับส่งคำตอบ
# และตำแหน่งไฟล์ผลลัพธ์ที่จะถูกสร้างหลังประมวลผลเสร็จ
IMAGES_DIR = Path("/kaggle/input/YOUR_DATASET/images")
TEMPLATE_PATH = Path("/kaggle/input/YOUR_DATASET/submission_template_v3.csv")
OUTPUT_CSV = Path("/kaggle/working/submission.csv")
DEBUG_CSV = Path("/kaggle/working/debug_rows.csv")

# โมเดล OCR ที่ใช้ และจำนวน token สูงสุดที่อนุญาตให้โมเดลสร้าง
MODEL_ID = "typhoon-ai/typhoon-ocr1.5-2b"
MAX_NEW_TOKENS = 3072

# ตัวเลือกสำหรับ debug
# ถ้าต้องการรันเฉพาะบางเอกสาร ให้ใส่ doc_id ลงไปใน set นี้
ONLY_DOC_IDS: set[str] | None = None
# ONLY_DOC_IDS = {"constituency_10_10", "party_list_10_13"}

# ถ้าต้องการจำกัดจำนวนเอกสารเพื่อทดสอบอย่างรวดเร็ว ให้กำหนดจำนวนที่นี่
LIMIT_DOCS: int | None = None
# LIMIT_DOCS = 3


## 2 : ฟังก์ชันเตรียมข้อมูลและแยกข้อความจากผล OCR

In [ ]:
# ฟังก์ชันช่วยปรับข้อความให้อยู่ในรูปแบบมาตรฐาน

# ใช้แปลงเลขไทยให้เป็นเลขอารบิก เพื่อให้ประมวลผลต่อได้ง่าย
THAI_DIGITS = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")

# ตารางแทนค่าตัวอักษรที่ OCR มักอ่านสลับกับตัวเลข
AMBIGUOUS_DIGITS = {
    "O": "0",
    "o": "0",
    "D": "0",
    "I": "1",
    "l": "1",
    "|": "1",
    "S": "5",
    "s": "5",
    "B": "8",
    "g": "9",
}

# ใช้ค้นหา token ที่มีแนวโน้มเป็นตัวเลขคะแนนเสียง
VOTE_TOKEN_PATTERN = re.compile(r"[0-9๐๑๒๓๔๕๖๗๘๙OoIlSBg|,./]+")


def normalize_vote_text(text: str) -> str:
    """
    ทำความสะอาดข้อความที่ควรเป็นคะแนนเสียง
    เช่น แปลงเลขไทย ตัด comma ช่องว่าง และแก้ตัวอักษรที่ OCR อ่านคล้ายตัวเลข
    """
    value = (text or "").translate(THAI_DIGITS)
    value = value.replace(",", "").replace(" ", "").replace("\n", "")
    value = "".join(AMBIGUOUS_DIGITS.get(ch, ch) for ch in value)
    value = re.sub(r"[^0-9]", "", value)
    return value


def extract_vote_numeric_token(text: str) -> str:
    """
    ดึง token ที่เป็นตัวเลขคะแนนจากข้อความ
    โดยพยายามใช้ส่วนหน้าของวงเล็บก่อน หากมีวงเล็บปนอยู่ในข้อความ
    """
    raw = " ".join((text or "").split())
    if not raw:
        return ""

    preferred_segments = [
        raw.split("(", 1)[0].split("（", 1)[0].strip(),
        raw,
    ]

    for segment in preferred_segments:
        if not segment:
            continue
        matches = VOTE_TOKEN_PATTERN.findall(segment)
        normalized = [normalize_vote_text(m) for m in matches]
        normalized = [m for m in normalized if m]
        if normalized:
            return normalized[0]

    return ""


def extract_candidate_number(text: str) -> int | None:
    """
    ใช้ดึงหมายเลขผู้สมัครจากข้อความ
    จำกัดช่วงไว้ที่ 1-99 เพื่อลด false positive จาก OCR
    """
    value = normalize_vote_text(text)
    if not value:
        return None

    try:
        number = int(value)
    except ValueError:
        return None

    if 1 <= number <= 99:
        return number
    return None


In [ ]:
# ฟังก์ชันสำหรับแยกผลลัพธ์ OCR ออกมาเป็นแถวข้อมูล

def split_party_and_vote(text: str) -> tuple[str, str]:
    """
    แยกข้อความออกเป็น 2 ส่วน:
    1) ชื่อพรรคหรือข้อความฝั่งซ้าย
    2) คะแนนเสียงที่อยู่ด้านท้ายข้อความ
    """
    raw = " ".join((text or "").split()).strip()
    if not raw:
        return "", ""

    match = re.search(r"[0-9๐๑๒๓๔๕๖๗๘๙OoIlSBg|,./]+", raw)
    if not match:
        return raw, ""

    party_text = raw[: match.start()].strip(" |:-")
    vote = extract_vote_numeric_token(raw[match.start() :])
    return party_text, vote


def parse_html_table_rows_in_order(text: str) -> list[dict[str, str]]:
    """
    กรณีโมเดลคืนผลเป็น HTML table
    ฟังก์ชันนี้จะอ่านทีละ <tr> แล้วพยายามแยกข้อมูลพรรคและคะแนนตามลำดับแถว
    """
    if "<table" not in text.lower() and "<tr" not in text.lower():
        return []

    row_matches = re.findall(r"<tr[^>]*>(.*?)</tr>", text, flags=re.IGNORECASE | re.DOTALL)
    if not row_matches:
        return []

    parsed_rows: list[dict[str, str]] = []

    for row_html in row_matches:
        cells = re.findall(r"<t[dh][^>]*>(.*?)</t[dh]>", row_html, flags=re.IGNORECASE | re.DOTALL)
        if not cells:
            continue

        plain_cells: list[str] = []
        for cell in cells:
            cell_text = re.sub(r"<[^>]+>", " ", cell)
            cell_text = html.unescape(cell_text)
            cell_text = " ".join(cell_text.split()).strip()
            plain_cells.append(cell_text)

        joined = " | ".join(cell for cell in plain_cells if cell)
        if not joined:
            continue

        # ข้ามแถวหัวตารางและแถวสรุปรวม
        if "รวมคะแนนทั้งสิ้น" in joined:
            continue
        if "ได้คะแนน" in joined or "หมายเลข" in joined or "ชื่อตัว" in joined:
            continue

        # กรณี cell ครบลักษณะตาราง 4 ช่องขึ้นไป:
        # [หมายเลขผู้สมัคร, ชื่อผู้สมัคร, พรรค, คะแนน]
        if len(plain_cells) >= 4:
            candidate_number = extract_candidate_number(plain_cells[0])
            if candidate_number is None:
                continue

            party_text = plain_cells[2].strip()
            vote = extract_vote_numeric_token(plain_cells[3])

            if vote:
                parsed_rows.append(
                    {
                        "party_text_raw": party_text,
                        "vote_raw": vote,
                    }
                )
            continue

        # กรณี cell ไม่ครบ ให้พยายามแยกจาก cell สุดท้ายแทน
        non_empty_cells = [cell for cell in plain_cells if cell]
        if not non_empty_cells:
            continue

        party_text, vote = split_party_and_vote(non_empty_cells[-1])
        if vote:
            parsed_rows.append(
                {
                    "party_text_raw": party_text,
                    "vote_raw": vote,
                }
            )

    return parsed_rows


def parse_table_rows(raw_text: str) -> list[dict[str, str]]:
    """
    พยายาม parse ข้อความจาก OCR ให้กลายเป็นรายการแถวข้อมูล
    โดยจะลอง HTML table ก่อน และถ้าไม่สำเร็จจึง fallback เป็นการอ่านทีละบรรทัด
    """
    rows = parse_html_table_rows_in_order(raw_text)
    if rows:
        return rows

    parsed_rows: list[dict[str, str]] = []
    for line in [x.strip() for x in raw_text.splitlines() if x.strip()]:
        if "รวมคะแนนทั้งสิ้น" in line:
            continue
        if "ได้คะแนน" in line or "หมายเลข" in line or "ชื่อตัว" in line:
            continue

        party_text, vote = split_party_and_vote(line)
        if vote:
            parsed_rows.append(
                {
                    "party_text_raw": party_text,
                    "vote_raw": vote,
                }
            )

    return parsed_rows


def extract_generated_text(result) -> str:
    """
    รองรับรูปแบบผลลัพธ์ที่ pipeline อาจคืนกลับมาได้หลายแบบ
    แล้วสรุปให้อยู่ในรูป string เดียวสำหรับนำไป parse ต่อ
    """
    if isinstance(result, str):
        return result

    if isinstance(result, dict):
        for key in ["generated_text", "text"]:
            value = result.get(key, "")
            if isinstance(value, str):
                return value
            if isinstance(value, list):
                return extract_generated_text(value)

    if isinstance(result, list):
        parts: list[str] = []
        for item in result:
            if isinstance(item, str):
                parts.append(item)
                continue

            if isinstance(item, dict):
                if "generated_text" in item:
                    parts.append(extract_generated_text(item["generated_text"]))
                    continue
                if "text" in item:
                    parts.append(extract_generated_text(item["text"]))
                    continue
                if "content" in item:
                    parts.append(extract_generated_text(item["content"]))
                    continue
                if item.get("type") == "text" and isinstance(item.get("text"), str):
                    parts.append(item["text"])
                    continue

            elif isinstance(item, list):
                parts.append(extract_generated_text(item))

        return "\n".join(part for part in parts if part)

    return str(result)


## 3 : ฟังก์ชันจัดกลุ่มเอกสาร และขั้นตอนประมวลผลหลัก

In [ ]:
# ฟังก์ชันช่วยจัดการชุดข้อมูลภาพและ id

def infer_doc_id(path: Path) -> str:
    """
    แยก doc_id จากชื่อไฟล์
    ตัวอย่าง constituency_10_10_page2.png -> constituency_10_10
    """
    name = path.stem
    match = re.match(r"^(.*)_page(\d+)$", name)
    if match:
        return match.group(1)
    return name


def infer_page_number(path: Path) -> int:
    """
    แยกเลขหน้าจากชื่อไฟล์ ถ้าไม่มีให้ถือว่าเป็นหน้า 1
    """
    name = path.stem
    match = re.match(r"^(.*)_page(\d+)$", name)
    if match:
        return int(match.group(2))
    return 1


def group_documents(images_dir: Path) -> dict[str, list[Path]]:
    """
    รวมไฟล์ภาพที่อยู่ในเอกสารเดียวกันไว้เป็นกลุ่มเดียว
    และเรียงลำดับตามเลขหน้า
    """
    groups: dict[str, list[Path]] = defaultdict(list)

    for path in sorted(images_dir.glob("*")):
        if path.suffix.lower() not in {".png", ".jpg", ".jpeg", ".webp"}:
            continue
        groups[infer_doc_id(path)].append(path)

    for doc_id in groups:
        groups[doc_id] = sorted(groups[doc_id], key=infer_page_number)

    return dict(groups)


def parse_id_fields(submission_id: str) -> tuple[str, int]:
    """
    แยก submission id ให้ได้:
    - doc_id
    - row_num
    เช่น constituency_10_10_3 -> (constituency_10_10, 3)
    """
    parts = submission_id.split("_")
    if len(parts) < 4:
        raise ValueError(f"Unexpected id format: {submission_id}")

    row_num = int(parts[-1])
    doc_id = "_".join(parts[:-1])
    return doc_id, row_num


In [ ]:
# ขั้นตอนการทำงานหลัก

def main() -> None:
    # ปิด warning บางประเภทที่ไม่จำเป็นต่อการใช้งานจริง
    warnings.filterwarnings("ignore")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

    # โหลด template และตรวจสอบว่ามีคอลัมน์สำคัญครบหรือไม่
    template = pd.read_csv(TEMPLATE_PATH, dtype=str).fillna("")
    required_cols = {"id", "votes"}
    missing = required_cols - set(template.columns)
    if missing:
        raise ValueError(f"Template missing columns: {missing}")

    # ถ้า template ไม่มีคอลัมน์ชื่อพรรค ให้เติมคอลัมน์ว่างไว้
    if "party_name" not in template.columns:
        template["party_name"] = ""

    # แยก id ออกเป็น doc_id และ row_num เพื่อใช้จับคู่แถวภายหลัง
    template[["doc_id", "row_num"]] = template["id"].apply(lambda x: pd.Series(parse_id_fields(x)))
    template["row_num"] = template["row_num"].astype(int)

    # จัดกลุ่ม template rows แยกตามเอกสาร และเรียงตาม row_num
    template_rows_by_doc = {
        doc_id: frame.sort_values("row_num").to_dict(orient="records")
        for doc_id, frame in template.groupby("doc_id", sort=False)
    }

    # สร้าง OCR pipeline
    pipe = pipeline(
        "image-text-to-text",
        model=MODEL_ID,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    # prompt ที่จะส่งให้โมเดล
    prompt = (
        "อ่านข้อความจากภาพตารางเลือกตั้งนี้ แล้วคืนผลเป็น HTML table เท่านั้น "
        "แต่ละแถวต้องเป็นข้อมูลผู้สมัครหรือพรรค 1 แถว "
        "ห้ามอธิบาย ห้ามคืน markdown ห้ามคืนข้อความอื่น"
    )

    # จัดกลุ่มภาพทั้งหมดเป็นเอกสาร
    documents = group_documents(IMAGES_DIR)

    # ตัวเลือกสำหรับ debug เฉพาะบาง doc_id
    if ONLY_DOC_IDS is not None:
        documents = {doc_id: pages for doc_id, pages in documents.items() if doc_id in ONLY_DOC_IDS}

    # ตัวเลือกสำหรับจำกัดจำนวนเอกสารที่ต้องการทดสอบ
    if LIMIT_DOCS is not None:
        documents = dict(list(documents.items())[:LIMIT_DOCS])

    submission_rows: list[dict[str, str]] = []
    debug_rows: list[dict[str, str | int]] = []

    # วนประมวลผลทีละเอกสาร
    for doc_idx, (doc_id, pages) in enumerate(tqdm(documents.items(), desc="Documents"), start=1):
        template_rows = template_rows_by_doc.get(doc_id, [])
        if not template_rows:
            continue

        print(f"\n[document {doc_idx}/{len(documents)}] {doc_id} pages={len(pages)}")
        parsed_rows_all: list[dict[str, str]] = []

        # วนประมวลผลทีละหน้าในเอกสาร
        for page_idx, page_path in enumerate(pages, start=1):
            print(f"  page {page_idx}: {page_path.name}")

            img = Image.open(page_path).convert("RGB")
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img},
                        {"type": "text", "text": prompt},
                    ],
                }
            ]

            result = pipe(text=messages, max_new_tokens=MAX_NEW_TOKENS)
            raw = extract_generated_text(result)
            print(f"    raw preview: {raw[:220]!r}")

            rows = parse_table_rows(raw)
            print(f"    parsed rows from page = {len(rows)}")
            parsed_rows_all.extend(rows)

            # เคลียร์หน่วยความจำระหว่างรัน
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print(f"  merged rows = {len(parsed_rows_all)}")

        # จับคู่ผล OCR กับ template ตามลำดับแถว
        for idx, tpl in enumerate(template_rows):
            row_num = int(tpl["row_num"])
            submission_id = tpl["id"]
            template_party = tpl.get("party_name", "")

            if idx < len(parsed_rows_all):
                ocr_party = parsed_rows_all[idx]["party_text_raw"]
                vote = normalize_vote_text(parsed_rows_all[idx]["vote_raw"])
                vote = vote if vote else "0"
            else:
                ocr_party = ""
                vote = "0"

            print(
                f"    map row_num={row_num} | id={submission_id} | "
                f"template_party={template_party} | ocr_party={ocr_party} | vote={vote}"
            )

            submission_rows.append(
                {
                    "id": submission_id,
                    "votes": vote,
                }
            )

            debug_rows.append(
                {
                    "id": submission_id,
                    "doc_id": doc_id,
                    "row_num": row_num,
                    "template_party_name": template_party,
                    "ocr_party_text_raw": ocr_party,
                    "vote_raw": vote,
                    "mapped_by": "row_order_to_id",
                    "page_count": len(pages),
                    "parsed_row_count": len(parsed_rows_all),
                }
            )

        # เซฟผล submission ชั่วคราวทุกครั้งหลังจบหนึ่งเอกสาร
        submission_df = pd.DataFrame(submission_rows)
        submission_df = template[["id"]].merge(submission_df, on="id", how="left")
        submission_df["votes"] = submission_df["votes"].fillna("0")
        submission_df.to_csv(OUTPUT_CSV, index=False)

    # เซฟไฟล์ debug หลังจบทั้งหมด
    debug_df = pd.DataFrame(debug_rows)
    debug_df.to_csv(DEBUG_CSV, index=False)

    print(f"\nsaved: {OUTPUT_CSV}")
    print(f"saved: {DEBUG_CSV}")


In [ ]:
# เรียกใช้งานโปรแกรมหลัก
if __name__ == "__main__":
    main()
